# Module 3: LangSmith — Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We observe and improve the **financial research agent** we built in Modules 1–2, working through the LangSmith lifecycle end to end:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the research agent, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, score the agent end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
4. **Online evaluations** — score every new trace as it is captured. Programmatic version + UI workflow.
5. **Annotation queues** — route runs flagged by eval scores to a human for review.
6. **Model choice & routing** — compare a larger vs. a smaller model on cost/efficacy, then route per-turn with Switchyard routing middleware.
7. **Engine** — automated failure-mode discovery across the trace corpus that ties the whole loop together.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))

# Suffix shared names (Hub repos, datasets, prompts) so attendees running the
# same workspace don't collide. Set WORKSHOP_USER in your .env to your handle;
# otherwise we fall back to the OS username.
import getpass
_user = os.environ.get("WORKSHOP_USER") or getpass.getuser()
def qualify(name: str) -> str:
    """Return a per-user name, e.g. qualify('financial-research') -> 'financial-research-alex'."""
    return f"{name}-{_user}"
print("Names qualified for user:", _user)

## The Agent: Financial Research

The agent we observe and improve in this module is the **financial research Deep Agent** from Modules 1–2. It plans a research request, delegates topic research to a subagent, synthesizes the findings, and writes a citation-backed note. It covers:

- **Equities & earnings** — summarize a company's latest quarter and what drove the print.
- **Macro & rates** — read the latest CPI, jobs, or FOMC release and explain the signal.
- **Sector / fixed income** — pull together market-moving news and frame the risks.

It's a supervisor + a `research-agent` subagent, wired to the model provider's **native web search** for gathering public sources, plus the built-in Deep Agent tools (`write_todos`, `task`, `write_file`, `read_file`).

Rather than treat it as a finished black box, we'll **compose it in three pieces** so each part is inspectable and versioned:

1. **Tools** — native web search + the built-in planning/file tools (wired in `agents/research_agent.py`).
2. **Memory** — an `AGENTS.md` operating manual: the agent's workflow and rules.
3. **Skills** — `earnings-summary` and `investor-note` output formats it can reach for.

We'll put the manual and skills in a **LangSmith Context Hub** repo — versioned in the Hub, pulled at build time — instead of pinning them to a local checkout. The next few cells walk through it. (We keep the agent lightweight — no HITL, no disk writes — so evaluation runs don't pause or leak files.)


### Compose it: inspect the manual and skills

The operating manual and skill files live under `agents/deep_agent/` — the same source of truth the deployable agent uses. Let's read them before we ship them to the Hub.


In [ ]:
from pathlib import Path

# The agent's memory + skills live alongside the deployable agent.
assets_dir = project_root / "agents" / "deep_agent"

# Memory: the operating manual.
agents_md = (assets_dir / "AGENTS.md").read_text()

# Skills: each is a folder with a SKILL.md (YAML frontmatter + instructions).
skill_files = {
    f"skills/{p.parent.name}/SKILL.md": p.read_text()
    for p in sorted(assets_dir.glob("skills/*/SKILL.md"))
}

print("AGENTS.md (first 12 lines):")
print("\n".join(agents_md.splitlines()[:12]))
print("\nSkills found:", list(skill_files))


### Compose it: push the manual + skills to Context Hub

**Context Hub** stores an agent's files (manual, skills, notes) in a versioned Hub repo, so the agent can pull them at build time instead of reading a local checkout. We push `AGENTS.md` and the skill files as one repo; each `push_agent` call is a new commit, so the manual and skills get the same version history as prompts do.


In [ ]:
from langsmith.schemas import FileEntry

context_repo = f"-/{qualify('financial-research-context')}"   # "-" = your default workspace tenant; per-user repo

# One file entry per path: the manual plus every skill's SKILL.md.
files = {"AGENTS.md": FileEntry(content=agents_md)}
files.update({path: FileEntry(content=body) for path, body in skill_files.items()})

commit_url = client.push_agent(
    context_repo,
    files=files,
    description="Financial research analyst manual + skills",
)
print("Context Hub commit (click to open):", commit_url)


### Compose it: build the agent from Context Hub

Now build the agent with `context_repo=` set. `build_research_agent` pulls that repo, mounts its `AGENTS.md` as the agent's memory and its `skills/` as the agent's skills — so the manual and skills come from the Hub, not the local disk. We build once and reuse this `agent` for the evals below.


In [ ]:
from agents.research_agent import build_research_agent

# Build once against the Context Hub repo we just pushed.
agent = build_research_agent(context_repo=context_repo)

# Peek at the structure: tools + subagent, plus the skills & memory middleware
# that the manual and skills add to the graph.
print("Graph nodes:", list(agent.get_graph().nodes))

quick = agent.invoke(
    {"messages": [{"role": "user", "content": "In one sentence, what is the Federal Reserve's dual mandate? Search at most once."}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("\nSample answer:\n" + quick["messages"][-1].text)


### The shape of the agent

Before we observe or evaluate it, let's *see* the agent: its graph and the tools it can call. 
The supervisor delegates to the `research-agent` subagent via the `task` tool, which uses 
**native web search** to gather sources; the supervisor plans with `write_todos` and writes the 
note with `write_file`. The Mermaid diagram below is rendered live from the compiled graph.


In [ ]:
from IPython.display import Image, display

# Render the compiled agent graph. draw_mermaid_png() calls a remote renderer,
# so fall back to the Mermaid text diagram if that isn't reachable.
graph = agent.get_graph()
try:
    display(Image(graph.draw_mermaid_png()))
except Exception as e:
    print(f"(PNG render unavailable: {e})\n")
    print(graph.draw_mermaid())

# Inventory the tools the agent can call, so teams see its surface area.
# Deep Agents registers built-in planning/file tools; the research work runs
# through native web search inside the model call.
tool_names = sorted(getattr(n, "name", n) for n in graph.nodes if n not in {"__start__", "__end__"})
print("\nGraph nodes / tool surface:")
for n in tool_names:
    print(f"  - {n}")


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** an earnings-recap prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = qualify("earnings-recap-assistant")
prompt = ChatPromptTemplate([
    ("system",
     "You are an equity research assistant. "
     "Look up the company's latest quarter, then produce a concise earnings recap: "
     "(1) a one-line summary, (2) headline numbers (revenue, EPS, margin) with Y/Y, "
     "(3) three notes on what drove the print, and (4) two questions for the next call. "
     "Be factual; cite figures and flag any non-GAAP adjustments explicitly."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for a **separate** `create_agent` demo (named `recap_agent` so it doesn't disturb the composed agent above), running on the workshop's shared `model`. A small mock `lookup_earnings` tool lets the agent fetch the company's latest-quarter figures, so the full agent loop (model → tool → model) shows up in the trace.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_earnings(ticker: str) -> str:
    """Look up a company's latest-quarter headline earnings figures by ticker."""
    # Mock financials lookup — swap for your real market-data source in production.
    directory = {
        "AAPL": (
            "AAPL — fiscal Q3. Revenue $85.8B (+5% Y/Y). EPS (GAAP) $1.40 (+11% Y/Y). "
            "Operating margin 29.6% (+120bps Y/Y). Services at a record; iPhone roughly flat. "
            "No guidance change. No non-GAAP adjustments flagged."
        ),
    }
    return directory.get(ticker.upper(), f"No earnings record found for {ticker!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

# Name this demo agent `recap_agent` so it doesn't clobber the composed research
# `agent` we built above (the warm-up and evals reuse that one).
recap_agent = create_agent(model=model, tools=[lookup_earnings], system_prompt=system_prompt)

result = recap_agent.invoke(
    {"messages": [{"role": "user", "content": "Recap AAPL's latest quarter."}]}
)
print(result["messages"][-1].text)

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are an equity research assistant. "
     "Look up the company's latest quarter, then produce a concise earnings recap: "
     "(1) a one-line summary, (2) headline numbers (revenue, EPS, margin) with Y/Y, "
     "(3) three notes on what drove the print, (4) two questions for the next call, "
     "and (5) one risk to keep in mind. "
     "Be factual; cite figures and flag any non-GAAP adjustments explicitly."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt(f"{prompt_name}:<commit-hash>"),
# and tear down with client.delete_prompt(prompt_name) when you're done.

## Warm-up: generate a few traces

Before we look at tracing and querying, we first produce some traces.
We invoke the research agent from Module 1 (`agents/research_agent.py`) three times with **intentionally short** prompts —
each one says "search at most once" so the runs finish in a few seconds instead of a few minutes.

During development, the warm-up took approximately 9 seconds total (3.1s average per call). Expect comparable runtime.

In [ ]:
from agents.research_agent import build_research_agent

agent = build_research_agent()

warmup_prompts = [
    "In one sentence, what is the federal funds rate? Search at most once.",
    "In one sentence, what does the VIX measure? Search at most once.",
    "In one sentence, what is a high-yield bond? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


### (Optional) Give the agent a sandbox for deterministic financial math

> **Optional cells — skip them and the rest of the notebook still runs.** They need `langsmith[sandbox]` installed and a workspace with the Sandbox feature enabled.

The warm-up agent answers from the model and web search. But a financial research agent shouldn't *reason* its way through the parts of the job that must be **exact and auditable** — year-over-year growth, margin math, reconciling segment figures back to a reported total, CAGR. LLMs are unreliable at arithmetic and easy to talk into a plausible-but-wrong number; a sandbox isn't.

So we hand the agent a **LangSmith sandbox** as its execution backend. Now it can *write and run real Python* to compute the figures in a note — deterministic, reproducible, and (because tracing is on) every sandbox command shows up in the trace next to the model's reasoning. That's the split you want in production: the model decides *what* to compute and *how to explain it*; the sandbox produces the *ground-truth numbers*.

We do it in **two cells** so a slow image build doesn't collide with the agent run:

1. **Build the snapshot** (below) — this compiles the Docker image into a bootable snapshot. It's the slow step, so we give it a generous timeout and wait until it reports `ready` before moving on. Re-running it is cheap once the snapshot exists.
2. **Boot a sandbox from the ready snapshot and run the agent** (next cell) — fast, and safe to re-run on its own without rebuilding.

Concretely, we then hand the agent a set of quarterly figures for **APPLE (AAPL)** and ask it to compute the key metrics for an earnings note — Y/Y revenue growth, gross and operating margins, and a check that the segments sum to reported revenue — by executing code, not by guessing.

In [ ]:
# OPTIONAL — Cell 1 of 2: build the snapshot and wait until it is ready.
# Requires `langsmith[sandbox]`; run `uv add "langsmith[sandbox]"` if the import fails.
from langsmith.sandbox import SandboxClient

sandbox_client = SandboxClient()  # uses LANGSMITH_ENDPOINT + LANGSMITH_API_KEY
snapshot_name = qualify("earnings-math")

# `create_snapshot` blocks until the image build finishes. Building can take a
# minute or two the first time, so bump the default 60s timeout. `fs_capacity_bytes`
# is REQUIRED and is a real filesystem size in *bytes* — 2 GiB here (not 512 bytes,
# which is far too small to unpack a Python image and is what makes the build fail).
snapshot = sandbox_client.create_snapshot(
    name=snapshot_name,
    docker_image="python:3.12-slim",
    fs_capacity_bytes=2 * 1024**3,  # 2 GiB
    timeout=300,
)

# Defensive: create_snapshot already waits, but poll once more so this cell only
# succeeds on a genuinely ready snapshot (raises on 'failed' with the build reason).
snapshot = sandbox_client.wait_for_snapshot(snapshot.id, timeout=300)
print(f"Snapshot {snapshot.name!r} is {snapshot.status} (id={snapshot.id}).")


In [ ]:
# OPTIONAL — Cell 2 of 2: boot a sandbox from the ready snapshot and run the agent.
# Safe to re-run on its own; it reuses `sandbox_client` / `snapshot` from the cell above.
from deepagents.backends import LangSmithSandbox
from deepagents import create_deep_agent

# A tiny synthetic set of quarterly figures to work from. In production these
# would come from the filing/market-data source the research-agent pulls.
financials_json = json.dumps({
    "ticker": "AAPL",
    "period": "Q3 FY2025",
    "prior_year_period": "Q3 FY2024",
    "revenue_usd_m": 90753,
    "prior_year_revenue_usd_m": 85777,
    "cost_of_sales_usd_m": 49290,
    "operating_expenses_usd_m": 15000,
    "reported_gross_margin_usd_m": 41463,
    "segments_usd_m": {
        "iPhone": 44582,
        "Mac": 8046,
        "iPad": 6580,
        "Wearables, Home and Accessories": 7396,
        "Services": 24149,
    },
})

with sandbox_client.sandbox(snapshot_id=snapshot.id, timeout=120) as sb:
    # Wrap the live sandbox as a deepagents backend so the agent's file + shell
    # tools run *inside* it instead of on the notebook host.
    sandbox_agent = create_deep_agent(
        model=model,
        tools=[],
        system_prompt=(
            "You are a financial research agent. You have a Python sandbox. "
            "For anything that must be exact -- year-over-year growth, gross and operating "
            "margins, and reconciling segment revenue to the reported total -- WRITE AND RUN "
            "PYTHON. Never do the arithmetic in your head. Report each computed figure clearly, "
            "flag any reconciliation that does not tie out, then give a short plain-language "
            "summary. Do not give investment advice or price targets."
        ),
        backend=LangSmithSandbox(sb),
    )

    cfg = {"configurable": {"thread_id": str(uuid7())}}
    result = sandbox_agent.invoke(
        {"messages": [{"role": "user", "content":
            "Compute the earnings-note metrics for this quarter by executing code: "
            "year-over-year revenue growth (%), gross margin (%) and operating margin (%), "
            "and confirm the segment revenues sum to reported revenue (flag any gap).\n\n"
            f"{financials_json}"
        }]},
        config=cfg,
    )
    print(result["messages"][-1].text)

# Why this matters: the Y/Y growth (90753/85777 - 1), the margins, and the
# segment-to-total reconciliation are now produced by code the agent ran -- not
# by the model asserting them. The whole thing (model + each sandbox command) is
# one trace you can inspect, evaluate, and route to review just like Parts 2-4.

## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition is recorded in the tracing project — no code changes required. 
(The warm-up above already generated traces; this section retrieves them.)

We use `client.list_runs(...)` to query them.


In [ ]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)


In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`


In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset.
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)


### 3.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching.


In [ ]:
examples = [
    {
        "inputs": {"query": "Write a one-line market summary for today to /summary.txt"},
        "outputs": {
            "reference_answer": "A one-line market summary saved to /summary.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research the latest US CPI release and write a brief note to /cpi_note.md"},
        "outputs": {
            "reference_answer": "A short note on the latest US CPI release, written to /cpi_note.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short memo on rate-cut expectations to /memo.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short rate-cut expectations memo, written and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan a small research project on Apple's most recent quarter, then research it and write a note to /apple_note.md"},
        "outputs": {
            "reference_answer": "A planned research note on Apple's most recent quarter, written to /apple_note.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

dataset_name = "financial-research-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


### 3.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request?
We construct the **LLM-as-judge from primitives** to make each component explicit:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects


In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the agent and return its final response.

    We also concatenate any files the agent wrote into the response string so
    the judge can see actual content -- not just the agent's "saved!" message.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    response = result["messages"][-1].content

    file_dump = []
    for path, file_data in (result.get("files") or {}).items():
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        file_dump.append(f"--- {path} ---\n{content}")
    if file_dump:
        response += "\n\nFiles written:\n" + "\n\n".join(file_dump)

    return {"response": response}


In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant's response.

You'll see the user's request, the assistant's final response, and a RUBRIC describing what success looks like.

The rubric is NOT the expected response text -- it's the success criteria. Mark `score=True` if the assistant's response demonstrates that the criteria were met (e.g., it confirms the file was written, or shows the content that was saved). The assistant's wording doesn't need to match the rubric -- what matters is whether the underlying task was accomplished.

Mark `score=False` only if the response clearly missed the task, contained errors, or refused without good reason.
Give one short sentence of reasoning either way.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.


In [ ]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it is captured in the tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`.
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.


In [ ]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an assistant response satisfied the user's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-online-correctness",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


## Part 5. Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge.
Any run matching the filter is added to the queue automatically.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="modular-workshops-needs-review",
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


In [ ]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### 5.1 Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "In one sentence, what is the difference between a stock and a bond? Search at most once.",
    "In one sentence, what does the term \"duration\" mean in fixed income? Search at most once.",
    "In one sentence, what is a credit default swap? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


### 5.2 Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "web_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to remove them when no longer needed.


## Part 6. Model Choice & Routing

### Compare two models: efficacy vs. cost

A practical question for any team shipping an agent: **which model should power it?** A larger 
model is usually more capable but slower and pricier; a smaller model is cheaper and faster but 
may cut corners. LangSmith experiments make this an evidence-based decision instead of a guess.

We run the **same financial research agent** over the **same tasks** with two different-sized 
models, then compare:

- **Efficacy** — a strict `correctness` score from an LLM-as-judge (did the agent nail *every* part?).
- **Cost & speed** — total tokens and wall-clock latency per run, pulled back from the traces.

The tasks are deliberately **multi-step** — research several facts, then reason or do arithmetic 
over the results (summarize a quarter *and* compute a Y/Y change, compare two companies on the same 
metric, connect a macro release to a rate call, list several drivers with citations). That's exactly 
where a smaller model tends to skip a step or miscompute, so the strict judge separates the two 
models instead of scoring both 100%.

We compare **`claude-sonnet-4-6`** (larger/default) against **`claude-haiku-4-6`** (smaller/cheaper). 
Each run is a LangSmith experiment, so you can open both in the UI and diff them side by side.


In [ ]:
from langchain.chat_models import init_chat_model
from agents.research_agent import build_research_agent

# The two models to compare: larger/default vs. smaller/cheaper.
MODELS_TO_COMPARE = {
    "claude-sonnet-4-6": "large (default)",
    "claude-haiku-4-5": "small (cheaper)",
}

def make_model(model_name):
    # Route through the LangSmith Gateway, matching utils/models.py — so the
    # comparison authenticates the same way as the rest of the workshop.
    return init_chat_model(
        model=model_name,
        model_provider="anthropic",
        base_url="https://gateway.smith.langchain.com/anthropic",
        api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
    )

# Tasks chosen to SEPARATE the models: each needs several research steps plus
# reasoning/arithmetic over the results, where a smaller model tends to skip a
# step, miscompute, or stop early. The reference_answer is a SUCCESS RUBRIC
# listing the concrete elements a complete, correct answer must contain -- the
# judge checks every one.
cmp_examples = [
    {
        "inputs": {"query": (
            "Summarize Apple's most recent quarterly earnings AND state the year-over-year "
            "change in total revenue in percent. Give the reporting period."
        )},
        "outputs": {"reference_answer": (
            "Must (1) name the reporting period (e.g. a fiscal quarter), (2) give total revenue "
            "for that quarter and the year-ago quarter, and (3) compute the Y/Y revenue change in "
            "percent consistent with those two figures. All three elements must be present and the "
            "percentage must match the numbers cited."
        )},
    },
    {
        "inputs": {"query": (
            "Summarize the latest US CPI release, AND state whether it argues for the Fed cutting, "
            "holding, or hiking at the next meeting, AND give the headline year-over-year rate."
        )},
        "outputs": {"reference_answer": (
            "Must combine THREE things: (1) a short summary of the latest CPI print, (2) the "
            "headline year-over-year CPI rate, and (3) a cut/hold/hike read for the next FOMC "
            "meeting that is consistent with that rate. All three must be present and coherent."
        )},
    },
    {
        "inputs": {"query": (
            "Compare Microsoft and Alphabet on most-recent-quarter total revenue: which is larger, "
            "and by how much? Show both figures."
        )},
        "outputs": {"reference_answer": (
            "Must look up BOTH companies and give each one's most-recent-quarter total revenue, "
            "then state which is larger and the difference between the two figures. Both revenue "
            "figures AND the difference must be present and consistent."
        )},
    },
    {
        "inputs": {"query": (
            "What did the most recent FOMC statement decide on the federal funds target range, "
            "and list the two main reasons the Committee gave for that decision."
        )},
        "outputs": {"reference_answer": (
            "Must (1) state the most recent FOMC decision on the target range (hold/cut/hike and "
            "the level or change), and (2) list two distinct reasons the statement cited (e.g. "
            "inflation still above target, labor market balance). Both the decision and two "
            "reasons must be present and accurate to the statement."
        )},
    },
]

cmp_dataset_name = qualify("model-comparison-evals")
if client.has_dataset(dataset_name=cmp_dataset_name):
    client.delete_dataset(dataset_id=client.read_dataset(dataset_name=cmp_dataset_name).id)
cmp_dataset = client.create_dataset(cmp_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in cmp_examples],
    outputs=[e["outputs"] for e in cmp_examples],
    dataset_id=cmp_dataset.id,
)
print(f"Comparison dataset '{cmp_dataset_name}' ready with {len(cmp_examples)} examples.")


In [ ]:
# A STRICT correctness judge for the comparison. Unlike a lenient 'good enough'
# grader, this one fails the response if ANY required element is missing or any
# number is wrong -- which is what surfaces a smaller model skipping a step or
# miscomputing. A small model keeps judging cheap and consistent across runs.
class CmpGrade(TypedDict):
    """Strict score: does the response contain every required element, all correct?"""
    score: bool
    reasoning: str

cmp_judge_prompt = (
    "You are a STRICT grader for a financial research assistant.\n"
    "The success rubric lists the specific elements a complete, correct answer must "
    "contain (figures, periods, verdicts, totals, differences).\n\n"
    "Mark score=True ONLY IF the response includes EVERY required element AND every "
    "number/verdict is correct and internally consistent. Mark score=False if the "
    "response omits any required element, gets any number or read wrong, stops early, "
    "or only partially answers a multi-part request. Do not give credit for being 'close.'\n"
    "In one sentence, state exactly which required elements are present/correct and "
    "which are missing/wrong."
)
cmp_judge = make_model("claude-haiku-4-5").with_structured_output(CmpGrade)

def cmp_correctness(inputs, outputs, reference_outputs):
    grade = cmp_judge.invoke([
        SystemMessage(content=cmp_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric (required elements): {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


### Run one experiment per model
The target builds the agent on that model, and each row keeps the run (latency + token cost) and the correctness score.

In [ ]:
cmp_results = {}

for model_name, label in MODELS_TO_COMPARE.items():
    cmp_agent = build_research_agent(model=make_model(model_name))

    def run_on_model(inputs, _agent=cmp_agent):
        result = _agent.invoke(
            {"messages": [{"role": "user", "content": inputs["query"]}]},
            config={"configurable": {"thread_id": str(uuid7())}},
        )
        return {"response": result["messages"][-1].text}

    cmp_results[model_name] = client.evaluate(
        run_on_model,
        data=cmp_dataset_name,
        evaluators=[cmp_correctness],
        experiment_prefix=qualify(f"model-cmp-{model_name}"),
        metadata={"model": model_name, "size": label},
        max_concurrency=2,
    )
    print(f"{model_name} ({label}) -> {cmp_results[model_name].experiment_name}")


### Reading the result
On these multi-step tasks you should see the models **separate** — the 
larger model typically scores higher `correctness` because it completes every required element and 
gets the arithmetic right, while the smaller model more often drops a required figure, miscomputes a 
Y/Y change, or answers only part of a multi-part request. The table quantifies the trade: how much 
accuracy you'd give up to save on latency and tokens. If the smaller model *does* keep pace here, 
that's a strong signal it's good enough for this workload. Either way you now have evidence, and the 
tracing, datasets, judges, and online evals in this module let you keep making that call as the 
agent evolves.


In [ ]:
import time as _time
from statistics import mean

# Summarize each experiment: efficacy (correctness) + speed (latency) come
# straight from the result rows; cost (tokens) is read back from the traces,
# because token usage is aggregated server-side and isn't on the local run
# object. Tokens land a few seconds after the run, so we retry briefly.
def _tokens_by_experiment(experiment_name, retries=6, delay=5):
    for _ in range(retries):
        runs = list(client.list_runs(project_name=experiment_name, is_root=True))
        toks = [r.total_tokens for r in runs if r.total_tokens]
        if toks:
            return mean(toks)
        _time.sleep(delay)
    return None

def summarize(results):
    scores, latencies = [], []
    for row in results:
        for r in row["evaluation_results"]["results"]:
            if r.key == "correctness" and r.score is not None:
                scores.append(r.score)
        # RunTree exposes .latency (seconds); it has no token fields.
        if row["run"].latency is not None:
            latencies.append(row["run"].latency)
    return {
        "correctness": mean(scores) if scores else None,
        "avg_latency_s": mean(latencies) if latencies else None,
        "avg_tokens": _tokens_by_experiment(results.experiment_name),
    }

print(f"{'model':<16}{'size':<16}{'correctness':>12}{'avg latency':>14}{'avg tokens':>13}")
print("-" * 71)
for model_name, label in MODELS_TO_COMPARE.items():
    s = summarize(cmp_results[model_name])
    corr = f"{s['correctness']:.0%}" if s['correctness'] is not None else "n/a"
    lat = f"{s['avg_latency_s']:.1f}s" if s['avg_latency_s'] is not None else "n/a"
    tok = f"{s['avg_tokens']:.0f}" if s['avg_tokens'] is not None else "n/a"
    print(f"{model_name:<16}{label:<16}{corr:>12}{lat:>14}{tok:>13}")

print("\nTip: open both experiments in LangSmith and use the Compare view to diff")
print("per-example correctness, latency, and cost side by side.")


## Per-turn model routing with Switchyard middleware

The comparison above picks **one** model for the whole agent. But not every *turn* needs the same 
muscle: a quick lookup or a formatting turn can go to a cheap model, while a hard synthesis turn 
goes to the frontier model. **Routing** makes that choice per turn.

We use the **in-process `SwitchyardRoutingMiddleware`** (from NVIDIA NeMo Switchyard) — no separate 
service, no proxy. Routing happens inside the agent process, and every routed `AIMessage` carries 
its decision in `response_metadata["switchyard"]`. The middleware reads tool-call / tool-result 
signals in the message history to decide, so there's no extra judge call and ~0 added latency.

**What we'll do**
1. Compute the **cost tradeoff** *first* — it can rule routing out before you build anything.
2. Wire the **`SwitchyardRoutingMiddleware`** into a Deep Agent (capable + efficient targets).
3. Run it over a set of financial-research turns and inspect the routing decisions.
4. Tally where calls (and money) went.

> **Switchyard is pre-alpha.** APIs change fast. Pin versions. All sample tasks here are for the 
> workshop and are not investment advice.

### The workload: financial-research turns

We route over a small set of research turns spanning **easy** (a one-line definition), **medium** 
(a single-topic summary), and **hard** (multi-part synthesis with arithmetic). The mix is the point: 
routing pays off only when a meaningful share of turns are easy enough for the cheap model.


In [ ]:
from dataclasses import dataclass

@dataclass
class Task:
    id: str
    difficulty: str   # 'easy' | 'medium' | 'hard'
    prompt: str

# A small spread of financial-research turns. The mix matters: routing only
# pays off when enough turns are easy enough for the cheap model.
TASKS = [
    Task("def-fed-funds", "easy",
         "In one sentence, what is the federal funds rate?"),
    Task("def-vix", "easy",
         "In one sentence, what does the VIX measure?"),
    Task("summ-cpi", "medium",
         "Summarize the latest US CPI release in three sentences."),
    Task("summ-fomc", "medium",
         "Summarize the most recent FOMC statement in three sentences."),
    Task("synth-earn", "hard",
         "Summarize Apple's most recent quarter AND compute the year-over-year "
         "revenue change in percent, showing both revenue figures."),
    Task("synth-cmp", "hard",
         "Compare Microsoft and Alphabet on most-recent-quarter total revenue: "
         "which is larger and by how much? Show both figures."),
]

def as_messages(task):
    """Shape a Task into the messages payload the agent expects."""
    return [{"role": "user", "content": task.prompt}]

# --- Cost tradeoff, computed FIRST ---------------------------------------
# Routing helps only if (a) a meaningful share of turns can use the cheap
# model and (b) the cheap model is much cheaper. Blended cost per turn is:
#     cost = p_easy * cheap + (1 - p_easy) * capable
# Compare that against always using the capable model.
cheap_cost_per_1k = 0.001     # illustrative $/1k tokens for the efficient model
capable_cost_per_1k = 0.015   # illustrative $/1k tokens for the capable model
p_easy = sum(t.difficulty in ("easy", "medium") for t in TASKS) / len(TASKS)

blended = p_easy * cheap_cost_per_1k + (1 - p_easy) * capable_cost_per_1k
always_capable = capable_cost_per_1k
savings = 1 - blended / always_capable
print(f"Share of turns eligible for the cheap model: {p_easy:.0%}")
print(f"Blended cost/1k:  ${blended:.4f}   vs. always-capable ${always_capable:.4f}")
print(f"Projected savings if routing is accurate: {savings:.0%}")
print("\nIf that savings doesn't justify the added complexity, stop here.")


### Wire in the routing middleware

The middleware takes a **capable** target and an **efficient** target and picks per turn. It runs
in-process — every routed `AIMessage` carries its decision trace in
`response_metadata["switchyard"]`.

#### Install from source (middleware is not on PyPI)

```bash
# Python 3.12+ required.
git clone https://github.com/NVIDIA-NeMo/Switchyard.git
python -m pip install -e ./Switchyard

git clone https://github.com/langchain-ai/langchain-nvidia.git
python -m pip install -e "./langchain-nvidia/libs/switchyard[openrouter]"
```

The middleware routes models through **OpenRouter**, so set `OPENROUTER_API_KEY` and confirm
your account can access both configured models.

In [ ]:
# Argument order matters: capable target FIRST, efficient target SECOND.
try:
    from langchain_openrouter import ChatOpenRouter
    from switchyard.libsy import LlmTarget, algorithms
    from langchain_nvidia_switchyard import LangChainLlmClient, SwitchyardRoutingMiddleware

    efficient_model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning-30b-a3b")
    capable_model = ChatOpenRouter(model="anthropic/claude-opus-4.8")

    router = algorithms.stage_router(
        LlmTarget("capable", LangChainLlmClient(capable_model)),
        LlmTarget("efficient", LangChainLlmClient(efficient_model)),
        picker="efficient_first",
        confidence_threshold=0.5,
        recent_window=3,
    )
    middleware = SwitchyardRoutingMiddleware(router)
    MIDDLEWARE_READY = True
    print("Switchyard middleware ready (stage_router, efficient_first).")
except ImportError as e:
    MIDDLEWARE_READY = False
    print(f"Middleware not installed - see the install cell above. Import error:\n  {e}")

In [ ]:
from deepagents import create_deep_agent

if MIDDLEWARE_READY:
    # Deep Agents needs a base model; the middleware substitutes its own choice per call.
    # Reuse the efficient target to avoid constructing an unused third model.
    agent_mw = create_deep_agent(model=efficient_model, middleware=[middleware])

    # ainvoke is canonical in notebooks.
    task = next(t for t in TASKS if t.difficulty == "easy")
    result_mw = await agent_mw.ainvoke({"messages": as_messages(task)})
    print("TASK:", task.id, f"({task.difficulty})")
    print(result_mw["messages"][-1].content[:600])
else:
    print("Skipping - middleware not installed.")

### Inspect routing decisions

`response_metadata["switchyard"]` carries `selected_model` (final pick) and `decisions`
(full ordered trace when an algorithm decides more than once).

In [ ]:
from langchain.messages import AIMessage

if MIDDLEWARE_READY:
    msg = next(
        m for m in reversed(result_mw["messages"]) if isinstance(m, AIMessage)
    )
    routing = msg.response_metadata.get("switchyard", {})
    print("selected_model:", routing.get("selected_model"))
    print("decisions:     ", routing.get("decisions"))
else:
    print("Skipping - middleware not installed.")

In [ ]:
from collections import Counter

async def run_all_middleware(agent, tasks):
    picks = Counter()
    rows = []
    for t in tasks:
        res = await agent.ainvoke({"messages": as_messages(t)})
        msg = next(m for m in reversed(res["messages"]) if isinstance(m, AIMessage))
        sel = msg.response_metadata.get("switchyard", {}).get("selected_model", "unknown")
        picks[sel] += 1
        rows.append({"id": t.id, "difficulty": t.difficulty, "selected_model": sel})
    return rows, picks

if MIDDLEWARE_READY:
    rows_mw, picks = await run_all_middleware(agent_mw, TASKS)
    for r in rows_mw:
        print(r)
    print("\nSelected-model tally:", dict(picks))
else:
    print("Skipping - middleware not installed.")

---
## Part 7. Engine — Automated Agent Improvement (tying it together)

We've now traced the research agent, evaluated it offline and online, routed flagged runs to a queue, and weighed model choice. **Engine ties all of that into one continuous loop.**

Querying traces by hand (Part 2) works because the project is small. Production agents push past that quickly:

- **Agents are longer-running** — a single trace can span dozens of tool calls and multiple subagent invocations.
- **Multimodal inputs and outputs** — traces include images, audio, attachments, and structured documents alongside text.
- **Agents are proliferating** — a typical deployment runs many agents, each producing its own trace stream.

**Engine performs this trace inspection automatically and continuously.** It turns the trace corpus into a continuous-improvement workflow: it surfaces recurring issues, diagnoses their root cause, and guides you through fixing them and preventing them from coming back. (Docs: <https://docs.langchain.com/langsmith/engine>.)


### 7.1 What Engine does

Engine plugs into the full agent engineering lifecycle: **trace → recurring failure detected → root cause diagnosed → fix proposed → evaluator deployed → dataset examples generated.** It runs continuously against the project's traces (scanning every ~6 hours by default) and surfaces issues without requiring manual review.

Each issue ships with a proposed code or prompt fix you can apply as a pull request, a suggested evaluator to catch regressions on the same failure mode, and dataset examples with assertions you can promote into experiments — closing the loop back into your regression suite.


### 7.2 Enabling Engine

Two configuration steps:

1. **Organization Admin** enables Engine workspace-wide in org settings.
2. **Per project**, configure:
   - Optionally connect a **GitHub repository** so fix proposals can become pull requests directly.
   - Select **priority categories** — which failure types matter most.
   - Review the **auto-generated agent overview** document Engine produces; correct any misunderstandings before issue generation starts.

Once enabled, Engine populates an **Issues** list in the project as it identifies failure patterns.

<img src="../images/engine_config.png" alt="Engine configuration dialog with GitHub connection and priority categories" style="width: auto; max-height: 380px; border-radius: 8px;">


### 7.3 The closed loop

Each Engine issue includes four components:

- **Diagnosis** — the root cause, linked to the failing traces. Inspect the linked traces directly to confirm the diagnosis matches your read.
- **Proposed Fix** — code or prompt changes; openable as a PR with one click if a GitHub repo is connected.
- **Suggested Evaluator** — a check you can deploy as a new run rule to catch regressions on this failure mode going forward.
- **Add offline examples** — extract the failing runs as dataset entries for the experiment loop.

<img src="../images/engine_triage.png" alt="Engine triage view with the issues list" style="width: auto; max-height: 420px; border-radius: 8px;">

<img src="../images/engine_issue.png" alt="Engine issue detail with diagnosis, proposed fix, and suggested evaluator" style="width: auto; max-height: 420px; border-radius: 8px;">

Priority adjustments and feedback (accept / reject / refine) train Engine's understanding of which issues matter for this specific project — relevance improves over iterations.

**Engine and the rest of this notebook close the loop together:**

- **Suggested evaluators → Part 4** (online run rules — same `create_run_rule` helper).
- **Dataset examples with assertions → Part 3** (offline experiments — promote them directly onto a dataset).
- **Issues that need human review → Part 5** (annotation queues — same routing rule pattern).


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and pull prompts | Playground / Prompt Hub · `push_prompt` / `pull_prompt` |
| **Warm-up** | Generate a few traces with the research agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **5. Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |
| **6. Model choice & routing** | Compare models on cost/efficacy, then route per-turn | `client.evaluate` experiments · `SwitchyardRoutingMiddleware` |
| **7. Engine** | Automated failure-mode discovery + suggested fixes, evaluators, dataset examples | LangSmith UI (cloud) |

The full loop: trace → Engine surfaces patterns → online eval scores incoming runs → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.
